### Run Eddy Tracking

In [ ]:
import os
from glob import glob
from pathlib import Path

# Correspondances is the PET tracker
from utils.py_eddy_tracker.tracking import Correspondances


Constants / Configs

In [ ]:
LON_RANGE = (-81, -56)
LAT_RANGE = (29, 44)
DATE_RANGE = ("2025-01-01", "2025-05-01") # use None for no bound

DATA_DIR = Path('/Users/jerry/school/research/eddy-tracking/data/gulf_stream_20241001_20250701/swot_l4')
ANTICYCLONE_DIR = Path('/Users/jerry/school/research/eddy-tracking/outputs/gulf_stream_20241001_20250701/eddy_id/anticyclone')
CYCLONE_DIR = Path('/Users/jerry/school/research/eddy-tracking/outputs/gulf_stream_20241001_20250701/eddy_id/cyclone')

ANTICYCLONE_TRACKED_DIR = Path('/Users/jerry/school/research/eddy-tracking/outputs/gulf_stream_20241001_20250701/eddy_track/anticyclone')
CYCLONE_TRACKED_DIR = Path('/Users/jerry/school/research/eddy-tracking/outputs/gulf_stream_20241001_20250701/eddy_track/cyclone')

In [ ]:
anticyclone_files = sorted(glob(f"{ANTICYCLONE_DIR}/*.nc"))
cyclone_files = sorted(glob(f"{CYCLONE_DIR}/*.nc"))

Correspondances takes in sorted daily eddy id files and links eddies across consecutive days.

For each pair of consecutive days:
1. Compute a cost matrix between all eddies on day N and N+1
2. Finds optimal matching to minimize total cost
3. Rejects any matches beyond a max distance (125 km)
4. Links matched eddies into tracks

In [ ]:
# Instantiate Correspondances and perform tracking
# Assums input datasets is sorted
a_corr = Correspondances(
    datasets=anticyclone_files,
    virtual=3, # max consecutive missing days before track ends
)
a_corr.track()

# Merge into a single dataset with track IDs
a_corr.prepare_merging()
a_corr.longer_than(10) # keep tracks >= 10 days
a_tracked = a_corr.merge(raw_data=False) # apply scale factors and offsets when reading netcdf files (process after converting from compressed form)

# Do the same with cyclonic eddies
c_corr = Correspondances(
    datasets=cyclone_files,
    virtual=3
)
c_corr.track()

c_corr.prepare_merging()
c_corr.longer_than(10)
c_tracked = c_corr.merge(raw_data=False) # TrackEddiesObservations

Virtual gaps are timesteps where an eddy existed but wasn't detected by the identification algorithm. I.e., it was there, then disappeared for some time less than `virtual` threshold, then reappeared in detection.



In [ ]:
# .virtual is a 1D array; one bool per eddy detection across all tracks
a_virtual_mask = (a_tracked.time == 0)
a_tracked.virtual[:] = a_virtual_mask # force in-place change
a_tracked.filled_by_interpolation(a_tracked.virtual == 1)

a_fn = f"{str(ANTICYCLONE_TRACKED_DIR)}/anticyclone_tracks.zarr"
a_tracked.write_file(filename=a_fn)

c_virtual_mask = (c_tracked.time == 0)
c_tracked.virtual[:] = c_virtual_mask
c_tracked.filled_by_interpolation(c_tracked.virtual == 1)

c_fn = f"{str(CYCLONE_TRACKED_DIR)}/cyclone_tracks.zarr"
c_tracked.write_file(filename=c_fn)

### View Outputs of Eddy Tracking

In [ ]:
from utils.py_eddy_tracker.observations.tracking import TrackEddiesObservations

a = TrackEddiesObservations.load_file('/Users/jerry/school/research/eddy-tracking/outputs/gulf_stream_20241001_20250701/eddy_track/anticyclone/anticyclone_tracks.zarr')
c = TrackEddiesObservations.load_file('/Users/jerry/school/research/eddy-tracking/outputs/gulf_stream_20241001_20250701/eddy_track/cyclone/cyclone_tracks.zarr')

In [ ]:
a.position_filter(median_half_window=1, loess_half_window=5)
c.position_filter(median_half_window=1, loess_half_window=5)

All tracked eddy trajectories after position smoothing (median half-window = 1, Loess half-window = 5). Line color encodes track duration; trajectories are wrapped to [-180, 180] longitude.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np

def plot_tracks(ax, tracked, color):
    for i, _, _ in tracked.iter_on('track'):
        lon = (tracked.longitude[i] + 180) % 360 - 180
        lat = tracked.latitude[i]
        points = np.column_stack((lon, lat)).reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        n = len(segments)
        colors = np.zeros((n, 4))
        colors[:, :3] = color
        colors[:, 3] = np.linspace(0.15, 1.0, n)  # fade in over track lifetime
        lc = LineCollection(segments, colors=colors, linewidth=0.8)
        ax.add_collection(lc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))

plot_tracks(ax1, a, (0.8, 0.1, 0.1))
ax1.set_xlim(LON_RANGE)
ax1.set_ylim(LAT_RANGE)
ax1.set_aspect('equal')
ax1.set_title('Anticyclonic Tracks')
ax1.grid()

plot_tracks(ax2, c, (0.1, 0.2, 0.8))
ax2.set_xlim(LON_RANGE)
ax2.set_ylim(LAT_RANGE)
ax2.set_aspect('equal')
ax2.set_title('Cyclonic Tracks')
ax2.grid()

plt.tight_layout()
plt.show()

### Try `AreaTracker`

In [ ]:
from utils.py_eddy_tracker.featured_tracking.area_tracker import AreaTracker

How does changing `.position_filter()` and `virtual` and other parameters affect outputs of eddy tracking?

### Isolate Longest-Lived Eddies

Filter by lifetime and plot out their tracks. Pick out a few to obtain contours over time, and then obtain a lon/lat boolean mask for their contours.

Length refers to the number of observations, which is the count of times that a specific track id appears in the track array.

Time is days since 1950-01-01. Consider each individual eddy track to be non-overlapping subarrays of time. Within each subarray, time is strictly monotonically increasing.

In [ ]:
def calendar_days_per_track(tracked: TrackEddiesObservations) -> np.ndarray:
    """
    Compute the inclusive calendar duration (in days) of each track.

    Uses the contiguous ordering of the track array to find boundaries
    between tracks, then computes duration in days per track.

    Args:
        tracked: TrackEddiesObservations with contiguous track IDs.

    Returns:
        1D array of calendar days per track, indexed by track ID.
    """
    boundaries = np.flatnonzero(np.diff(tracked.track)) + 1
    boundaries = np.concatenate([[0], boundaries, [len(tracked.track)]])
    t_start = tracked.time[boundaries[:-1]]
    t_end = tracked.time[boundaries[1:] - 1]
    return t_end - t_start + 1

In [ ]:
a_cal_days = calendar_days_per_track(a)
c_cal_days = calendar_days_per_track(c)

print(
    f"polarity: anticyclonic\n"
    f"tracks: {len(a_cal_days)}\n"
    f"longest_days: {a_cal_days.max()}"
)
print(
    f"polarity: cyclonic\n"
    f"tracks: {len(c_cal_days)}\n"
    f"longest_days: {c_cal_days.max()}"
)

Find the longest-lived track IDs for each type.

In [ ]:
a_longest_ids = np.where(a_cal_days == a_cal_days.max())[0]
c_longest_ids = np.where(c_cal_days == c_cal_days.max())[0]

print(
    f"polarity: anticyclonic\n"
    f"longest_track_ids: {a_longest_ids}\n"
    f"longest_days: {a_cal_days.max()}"
)
print(
    f"polarity: cyclonic\n"
    f"longest_track_ids: {c_longest_ids}\n"
    f"longest_days: {c_cal_days.max()}"
)

Obtain new TrackEddiesObservations objects with only the longest lived eddies

In [ ]:
a_long = a.extract_ids(a_longest_ids)
c_long = c.extract_ids(c_longest_ids)

Trajectories of the longest-lived anticyclonic (left, red) and cyclonic (right, blue) eddies, colored by time progression. These eddies serve as case studies for subsequent PACE data overlay.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))

for ax, tracked, title, cmap_name in [
    (ax1, a_long, 'Longest Anticyclonic Tracks', 'Reds'),
    (ax2, c_long, 'Longest Cyclonic Tracks', 'Blues'),
]:
    cmap = plt.get_cmap(cmap_name)
    unique_ids = np.unique(tracked.track)
    for track_id in unique_ids:
        eddy_mask = tracked.track == track_id
        lon = (tracked.longitude[eddy_mask] + 180) % 360 - 180
        lat = tracked.latitude[eddy_mask]

        # Fade segments from transparent to opaque over the track lifetime
        points = np.column_stack((lon, lat)).reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        n = len(segments)
        seg_colors = np.zeros((n, 4))
        seg_colors[:, :3] = cmap(0.7)[:3]
        seg_colors[:, 3] = np.linspace(0.15, 1.0, n)
        lc = LineCollection(segments, colors=seg_colors, linewidth=1.5)
        ax.add_collection(lc)

        ax.plot(lon[0], lat[0], 'o', color=cmap(0.7), markersize=6)
        ax.plot(lon[-1], lat[-1], 's', color=cmap(0.9), markersize=6)

    ax.set_xlim(LON_RANGE)
    ax.set_ylim(LAT_RANGE)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('longest_lived_tracks.png', dpi=150, bbox_inches='tight')
plt.show()

Effective contour evolution of the longest-lived eddies over their tracking lifetime, showing how spatial extent and position change as each eddy propagates westward.

In [ ]:
def get_id_indices(tracked: TrackEddiesObservations, track_id: int) -> np.ndarray:
    """Obtain the row indices belonging to a specific track (e.g., for effective_contour_lon/lat)."""
    return np.where(tracked.track == track_id)[0]

a_longest_idxs = {} # map track id to row indices
c_longest_idxs = {}

for id in a_longest_ids:
    a_longest_idxs[id] = get_id_indices(a_long, id)
for id in c_longest_ids:
    c_longest_idxs[id] = get_id_indices(c_long, id)

# Plot contour evolution, subsampled weekly
SUBSAMPLE = 7

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))

for ax, tracked, idxs_dict, title, cmap_name in [
    (ax1, a_long, a_longest_idxs, 'Longest Anticyclonic Contours (weekly)', 'Reds'),
    (ax2, c_long, c_longest_idxs, 'Longest Cyclonic Contours (weekly)', 'Blues'),
]:
    cmap = plt.get_cmap(cmap_name)
    for track_id, rows in idxs_dict.items():
        sampled = rows[::SUBSAMPLE]
        n_sampled = len(sampled)
        for j, i in enumerate(sampled):
            lon = (tracked.contour_lon_e[i] + 180) % 360 - 180
            lat = tracked.contour_lat_e[i]

            # Close the polygon
            lon = np.append(lon, lon[0])
            lat = np.append(lat, lat[0])

            color = cmap(0.3 + 0.6 * j / max(n_sampled - 1, 1))
            ax.plot(lon, lat, color=color, linewidth=0.8, alpha=0.7)

            # Plot eddy center
            center_lon = (tracked.longitude[i] + 180) % 360 - 180
            center_lat = tracked.latitude[i]
            ax.plot(center_lon, center_lat, '.', color=color, markersize=4)

        # Label at first contour center
        first_i = rows[0]
        label_lon = (tracked.longitude[first_i] + 180) % 360 - 180
        label_lat = tracked.latitude[first_i]
        ax.annotate(
            str(track_id), (label_lon, label_lat),
            fontsize=9, fontweight='bold', color=cmap(0.9),
            textcoords='offset points', xytext=(5, 5),
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.7),
        )

    ax.set_xlim(LON_RANGE)
    ax.set_ylim(LAT_RANGE)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Readable subset of weekly contour evolution for cyclonic eddies, using the tracked contours from `outputs/.../eddy_track/cyclone`. The plot keeps only the longest-lived tracks, and only the top few are labeled and visually emphasized to reduce clutter.


In [ ]:
# Weekly contour evolution for a readable subset of long-lived cyclonic eddies
N_PLOT = 10
N_LABEL = 5
SUBSAMPLE = 7

# Rank cyclone track IDs by inclusive calendar duration, longest first
c_ranked_ids = sorted(range(len(c_cal_days)), key=lambda tid: (-c_cal_days[tid], tid))
cyclone_plot_ids = c_ranked_ids[:N_PLOT]
cyclone_label_ids = set(cyclone_plot_ids[:N_LABEL])
cyclone_plot_rows = {track_id: get_id_indices(c, track_id) for track_id in cyclone_plot_ids}

fig, ax = plt.subplots(figsize=(12, 7))
cmap = plt.get_cmap('Blues')

for rank, track_id in enumerate(cyclone_plot_ids):
    rows = cyclone_plot_rows[track_id]
    sampled = rows[::SUBSAMPLE]
    n_sampled = len(sampled)
    is_labeled = track_id in cyclone_label_ids

    for j, i in enumerate(sampled):
        lon = (c.contour_lon_e[i] + 180) % 360 - 180
        lat = c.contour_lat_e[i]

        lon = np.append(lon, lon[0])
        lat = np.append(lat, lat[0])

        color_level = 0.22 + 0.60 * j / max(n_sampled - 1, 1)
        color = cmap(color_level)
        linewidth = 1.05 if is_labeled else 0.7
        alpha = 0.82 if is_labeled else 0.38
        ax.plot(lon, lat, color=color, linewidth=linewidth, alpha=alpha, zorder=2)

        center_lon = (c.longitude[i] + 180) % 360 - 180
        center_lat = c.latitude[i]
        ax.plot(
            center_lon, center_lat, '.', color=color,
            markersize=4 if is_labeled else 2.5,
            alpha=0.9 if is_labeled else 0.55,
            zorder=3,
        )

    if is_labeled:
        label_i = sampled[len(sampled) // 2]
        label_lon = (c.longitude[label_i] + 180) % 360 - 180
        label_lat = c.latitude[label_i]
        ax.annotate(
            str(track_id), (label_lon, label_lat),
            fontsize=10,
            fontweight='bold',
            color=cmap(0.9),
            textcoords='offset points',
            xytext=(6, 6),
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='none', alpha=0.85),
            zorder=4,
        )

ax.set_xlim(LON_RANGE)
ax.set_ylim(LAT_RANGE)
ax.set_aspect('equal')
ax.set_title(f'Longest Cyclonic Contours (weekly, top {N_PLOT})')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(alpha=0.3)

out_fp = Path('visuals') / 'cyclonic_contours_weekly_subset.png'
out_fp.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(out_fp, dpi=150, bbox_inches='tight')
plt.show()

print(f'cyclone_track_ids_plotted: {cyclone_plot_ids}')
print(f'cyclone_track_ids_labeled: {sorted(cyclone_label_ids)}')
print(
    f'figure_path: {out_fp}\n'
    f'status: saved'
)



### Overlay PACE Data

Perform quality control methods.

Raw downloaded data should be read-only. 

Procedure:

Iterate through each PACE file and obtain the date from the filename. Look up long-lived eddy positions/contours on that date and check spatial overlap.

We only look at snapshots where the chosen eddy is fully within the PACE swath (all points forming contour are within swath bounds).

1. Build a polygon from the 50 (lon, lat) eddy contour points
2. Find PACE pixels inside of the contour
3. Apply QC flag mask
4. Compute QC fraction, which is `valid_mask.sum() / contains_mask.sum()`. If below 50%, then don't use this PACE data granule.

After identifying valid PACE pixels, SST and SSS come in. Ultimately, Rrs -> (n_samples, n_wavelengths), temp -> (n_samples), sal -> (n_samples). (lon, lat, time) gets collapsed into individual points.

In [ ]:
# Define target eddy
# NOTE: c_eddy_id must be updated after re-running tracking with the new date range.
# The track IDs will change since tracking produces different results with more data.
print(c_longest_ids)
c_eddy_id = 20  # chosen eddy id — UPDATE after re-tracking

In [ ]:
from matplotlib import path
import xarray as xr


def extract_eddy_pixels(
    lon: np.ndarray,
    lat: np.ndarray,
    rrs: np.ndarray,
    contour_lon: np.ndarray,
    contour_lat: np.ndarray,
    min_coverage: float = 0.5,
) -> dict | None:
    """
    Extract QC-filtered Rrs pixels within an eddy contour from L3 gridded data.

    L3 data has QC pre-applied during processing: flagged pixels are NaN.
    A pixel is "valid" if it falls inside the eddy contour polygon and all
    its Rrs wavelengths are finite (not NaN).

    Args:
        lon: 1D longitude array from L3 grid.
        lat: 1D latitude array from L3 grid.
        rrs: 3D array (lat, lon, wavelength) of Rrs values.
        contour_lon: 1D array of eddy contour longitudes.
        contour_lat: 1D array of eddy contour latitudes.
        min_coverage: Minimum fraction of valid/inside pixels required.

    Returns:
        Dict with 'rrs', 'lon', 'lat', 'qc_fraction', or None if coverage < min_coverage.
    """
    lon2d, lat2d = np.meshgrid(lon, lat)
    N = lon2d.size
    lon_flat = lon2d.ravel()
    lat_flat = lat2d.ravel()
    rrs_flat = rrs.reshape(N, -1)  # (n_gridcells, n_wavelengths)

    polygon = path.Path(np.column_stack([contour_lon, contour_lat]))
    points = np.column_stack([lon_flat, lat_flat])

    inside = polygon.contains_points(points)

    # L3 QC: NaN means flagged. Valid = inside contour AND all wavelengths finite.
    all_finite = np.all(np.isfinite(rrs_flat), axis=1)
    valid = inside & all_finite

    n_inside = np.sum(inside)
    n_valid = np.sum(valid)

    if n_inside == 0:
        return None

    qc_fraction = n_valid / n_inside
    if qc_fraction < min_coverage:
        return None

    return {
        'rrs': rrs_flat[valid],
        'lon': lon_flat[valid],
        'lat': lat_flat[valid],
        'qc_fraction': float(qc_fraction),
    }

In [ ]:
# Build a table of each valid pixel from every L3 daily PACE file
# which covers the target eddy on that date.
# Columns: track_id, date, lon, lat, qc_fraction, Rrs_<wavelength>...

import datetime as dt
import pandas as pd

PACE_DIR = Path('/Users/jerry/school/research/eddy-tracking/data/gulf_stream_20241001_20250701/pace_l3')
MIN_COVERAGE = 0.8

rrs_fp = Path('data') / f'eddy_{c_eddy_id}_rrs.csv'

if rrs_fp.exists():
    inputs_df = pd.read_csv(rrs_fp, parse_dates=['date'])
    print(
        f"status: loaded_cached_csv\n"
        f"rows: {len(inputs_df)}"
    )
else:
    # Eddy contour data for the chosen track
    # PET uses 0-360 longitude; PACE uses -180 to 180, so convert here
    eddy_mask = c_long.track == c_eddy_id
    eddy_lons = (c_long.contour_lon_e[eddy_mask] + 180) % 360 - 180
    eddy_lats = c_long.contour_lat_e[eddy_mask]
    eddy_times = c_long.time[eddy_mask]
    times_set = set(eddy_times)

    # Read wavelengths from first L3 file
    pace_files = sorted(glob(f"{PACE_DIR}/*.nc"))
    with xr.open_dataset(pace_files[0]) as sample:
        wavelengths = sample.coords['wavelength'].values.astype(int)
    np.save('data/pace_wavelengths.npy', wavelengths)
    rrs_cols = [f'Rrs_{w}' for w in wavelengths]

    base = dt.datetime(1950, 1, 1)
    row_chunks = []

    for fp in pace_files:
        # L3 daily filename: PACE_OCI.YYYYMMDD.L3m.DAY.RRS...
        fn = Path(fp).name
        pace_date = dt.datetime.strptime(fn.split('.')[1], '%Y%m%d')
        pace_time_pet = (pace_date - base).days

        # Match up so that rrs data is on the same day as eddy
        if pace_time_pet not in times_set:
            continue

        same_day_idx = np.flatnonzero(eddy_times == pace_time_pet)[0]

        with xr.open_dataset(fp) as ds:
            result = extract_eddy_pixels(
                ds['lon'].values,
                ds['lat'].values,
                ds['Rrs'].values,
                eddy_lons[same_day_idx],
                eddy_lats[same_day_idx],
                min_coverage=MIN_COVERAGE,
            )

        if result is None:
            continue

        n = len(result['lon'])
        chunk = np.column_stack([
            np.full(n, c_eddy_id),
            np.full(n, pace_time_pet),
            result['lon'],
            result['lat'],
            np.full(n, result['qc_fraction']),
            result['rrs'],
        ])
        row_chunks.append(chunk)
        print(
            f"date: {pace_date.date()}\n"
            f"valid_points: {n}\n"
            f"qc_fraction: {result['qc_fraction']:.3f}"
        )

    if not row_chunks:
        raise RuntimeError("No valid PACE observations found for this eddy")

    # Save the data to disk
    all_data = np.vstack(row_chunks)
    columns = ['track_id', 'date', 'lon', 'lat', 'qc_fraction'] + rrs_cols

    inputs_df = pd.DataFrame(all_data, columns=columns)
    inputs_df['track_id'] = inputs_df['track_id'].astype(int)
    inputs_df['date'] = pd.Timestamp('1950-01-01') + pd.to_timedelta(inputs_df['date'], unit='D')

    print(
        f"total_pixels: {len(inputs_df)}\n"
        f"data_granules: {len(row_chunks)}"
    )
    print(inputs_df.head())

    inputs_df.to_csv(rrs_fp, index=False)

Now that we have Rrs, we need to pre-process it for the SDP model. QC filtering has been done already. Next, we need to:
1. Interpolate to 1 nm (at an padded range of 396-704 nm)
2. 5 nm moving mean smoothing
3. Trim the edges 4 nm on either side

In [ ]:
from utils.sdp.preprocessing import preprocess_rrs_batch

wavelengths = np.load('data/pace_wavelengths.npy')
rrs_cols = [col for col in inputs_df.columns if col.startswith('Rrs_')]
rrs_raw = inputs_df[rrs_cols].values  # shape: (n_pixels, n_wavelengths)

# Preprocess: interpolate to 1 nm, smooth with 5 nm moving mean, trim edges
# wavelengths is [400, 700]
# rrs_preprocessed is table of observations with interpolated rrs
wavelengths, rrs_processed = preprocess_rrs_batch(wavelengths, rrs_raw)

print(f"input_shape: {rrs_raw.shape}")
print(f"output_shape: {rrs_processed.shape}")

# Check for all NaN rows
n_invalid = np.isnan(rrs_processed).any(axis=1).sum()
if n_invalid > 0:
    raise ValueError("error")

# If this runs properly, then there should be 0 NaN values in Rrs

Now that we have the matched table of PACE and eddy, obtain SSS and SST at each (lon, lat, date) to add matched `temp` and `sal` columns.

In [ ]:
from utils.sdp.ancillary import load_sst_dataset, load_sss_dataset, sample_ancillary

SST_DIR = Path('data/gulf_stream_20241001_20250701/sst')
SSS_DIR = Path('data/gulf_stream_20241001_20250701/sss')

# Load all gridded files into DataArrays with a time dimension
sst_da = load_sst_dataset(SST_DIR)
sss_da = load_sss_dataset(SSS_DIR)

# Sample nearest SST/SSS for every (date, lat, lon) in the pigments table
sst_vals, sss_vals = sample_ancillary(
    sst_da, sss_da,
    lons=inputs_df["lon"].values,
    lats=inputs_df["lat"].values,
    times=pd.to_datetime(inputs_df["date"]).values,
)
inputs_df['sst'] = sst_vals
inputs_df['sss'] = sss_vals

Obtain residuals and run the PCR for pigments

In [ ]:

from utils.sdp import run_sdp

pigments_fp = Path('data') / f'eddy_{c_eddy_id}_pigments.csv'

if pigments_fp.exists():
    pigments_df = pd.read_csv(pigments_fp)
    print(
        f"status: loaded_saved_pigments\n"
        f"shape: {pigments_df.shape}"
    )
else:
    # Build Rrs DataFrame with integer wavelength columns which is what's expected as model input
    wl_int = wavelengths.astype(int)
    rrs_df = pd.DataFrame(rrs_processed, columns=wl_int)

    # Run SDP model: GSM inversion -> residuals -> 2nd derivative -> PCR
    pigments_df = run_sdp(
        rrs=rrs_df,
        wl=wavelengths,
        sst=inputs_df['sst'].to_numpy(),
        sss=inputs_df['sss'].to_numpy(),
    )

    pigments_df.to_csv(pigments_fp, index=False)
    print(
        f"status: saved_pigments\n"
        f"path: {pigments_fp}"
    )

print(f"pigment_predictions_shape: {pigments_df.shape}")
print(pigments_df)

In [ ]:
# Add time and location context to pigment (rows align with inputs_df)
pigments_df['lon']  = inputs_df['lon'].values
pigments_df['lat']  = inputs_df['lat'].values
pigments_df['date'] = inputs_df['date'].values

pigments_df.info()

### Check whether the values predicted from SDP model make sense.

First check whether the relationship between predicted values makes sense.

Use DPA framework to estimate the contribution of each major phytoplankton group to total chlorophyll a, using significant accessory pigments with empiricle weights.

In [ ]:
# Uitz diagnostic pigment analysis
# TChla = 1.41*Fuco + 1.41*Perid + 1.27*HexFuco + 0.35*ButFuco + 0.60*Allo  + 1.01*TChlb + 0.86*Zea

DPA_WEIGHTS = {
    'Fuco':    1.41,
    'Perid':   1.41,
    'HexFuco': 1.27,
    'ButFuco': 0.35,
    'Allo':    0.60,
    'MV chlb': 1.01, # use in place of TChlb
    'Zea':     0.86,
}

tchla_pred = sum(pigments_df[pig] * w for pig, w in DPA_WEIGHTS.items())
diffs = tchla_pred - pigments_df['T chla']

# Build a per-day comparison
rows = []
dates = pigments_df['date'].dt.date
unique_dates = sorted(dates.unique())

for d in unique_dates:
    mask = (dates == d)

    row = {
        'date': d,
        'n_pixels': mask.sum(),
        'mean_chla_sdp': pigments_df.loc[mask, 'T chla'].mean(),
        'mean_chla_dpa': tchla_pred[mask].mean(),
        'median_chla_sdp': pigments_df.loc[mask, 'T chla'].median(),
        'median_chla_dpa': tchla_pred[mask].median()
    }
    rows.append(row)

dpa_df = pd.DataFrame(rows)
print(dpa_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, sdp_col, dpa_col, stat in [
    (ax1, 'mean_chla_sdp', 'mean_chla_dpa', 'Mean'),
    (ax2, 'median_chla_sdp', 'median_chla_dpa', 'Median'),
]:
    ax.scatter(dpa_df[sdp_col], dpa_df[dpa_col], s=50, zorder=3)

    # 1:1 reference line
    lim = max(dpa_df[sdp_col].max(), dpa_df[dpa_col].max()) * 1.15
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='1:1 line')

    ax.set(xlabel=f'{stat} TChla, SDP (mg/m³)',
           ylabel=f'{stat} TChla, DPA (mg/m³)',
           title=f'Daily {stat}: SDP vs DPA',
           xlim=(0, lim), ylim=(0, lim))
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Check the fraction of zeros after running the model, since negative values are clipped to zero.

In [ ]:
# Fraction of zeros per pigment
# Negative SDP predictions are clipped to 0, so a high zero fraction signals
# frequent unphysical (negative) model outputs for that pigment.

pigment_cols = [c for c in pigments_df.columns if c not in ['lon', 'lat', 'date']]
zero_frac = (pigments_df[pigment_cols] == 0).mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(zero_frac)), zero_frac.values, color='steelblue')
ax.set_xticks(range(len(zero_frac)))
ax.set_xticklabels(zero_frac.index, rotation=45, ha='right')
ax.set(ylabel='Fraction of pixels = 0', title='Zero Fraction per Pigment', ylim=(0, 0.5))
ax.grid(alpha=0.3, axis='y')

# Annotate each bar with the percentage
for i, (name, frac) in enumerate(zero_frac.items()):
    ax.text(i, frac + 0.01, f'{frac:.1%}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


Check pigment ratio to total chlorophyll a and whether that matches with literature.

In [ ]:
# Pigment-to-TChla ratios vs. literature ranges. 
#
# Sources:
#   [1] Kramer & Siegel (2019) https://pmc.ncbi.nlm.nih.gov/articles/PMC7043335/
#   [2] Kramer et al. (2020) https://doi.org/10.3389/fmars.2020.00215
#   [3] Goericke & Repeta (1992) DVchla in subtropical N. Atlantic
#   [4] Higgins et al. (2011) Phytoplankton Pigments, Cambridge Univ. Press
#   [5] Aiken et al. (2009) https://doi.org/10.1016/j.dsr2.2008.09.017

LITERATURE = {
    'Perid':     {'range': (0.01, 0.05), 'group': 'Dinoflagellates'},    # regional [1]: BOUS 0.019, CAR 0.023; global 0.017
    'Fuco':      {'range': (0.05, 0.30), 'group': 'Diatoms'},           # regional [1]: BOUS 0.056, CAR 0.065; global 0.140
    'Zea':       {'range': (0.05, 0.30), 'group': 'Cyanobacteria'},     # regional [1]: BOUS 0.134, CAR 0.294
    'DV chla':   {'range': (0.10, 0.40), 'group': 'Prochlorococcus'},   # regional [3]: up to 40% of TChla, subtropical N. Atl
    'chl c3':    {'range': (0.02, 0.15), 'group': 'Haptophytes'},       # global [4,5]: CHEMTAX ratio matrices
    'HexFuco':   {'range': (0.10, 0.40), 'group': 'Haptophytes'},       # regional [1]: BOUS 0.253, CAR 0.102
    'MV chlb':   {'range': (0.00, 0.15), 'group': 'Green algae'},       # regional [1]: BOUS 0.000, CAR 0.046; global 0.045
}

# Compute observed ratios (exclude TChla == 0 pixels)
valid = pigments_df['T chla'] > 0
ratios = {}
for pig in LITERATURE:
    ratios[pig] = pigments_df.loc[valid, pig] / pigments_df.loc[valid, 'T chla']

# --- Summary table ---
print(f"{'Pigment':>10s}  {'Group':<18s}  {'Model Median':>13s}  {'Model IQR':>16s}  "
      f"{'Range from Lit.':>16s}")
print('-' * 80)

for pig, info in LITERATURE.items():
    r = ratios[pig]
    med = r.median()
    q25, q75 = r.quantile(0.25), r.quantile(0.75)
    lo, hi = info['range']

    print(f'{pig:>10s}  {info["group"]:<18s}  {med:13.3f}  [{q25:.3f}, {q75:.3f}]  '
          f'[{lo:.3f}, {hi:.2f}]')

# --- Bar chart: observed median vs literature range ---
fig, ax = plt.subplots(figsize=(12, 5))
names = list(LITERATURE.keys())
x = np.arange(len(names))

lows = [LITERATURE[p]['range'][0] for p in names]
highs = [LITERATURE[p]['range'][1] for p in names]
ax.bar(x, [h - l for h, l in zip(highs, lows)], bottom=lows,
       width=0.6, color='lightgreen', alpha=0.5, label='Range from literature')

medians = [ratios[p].median() for p in names]
q25s = [ratios[p].quantile(0.25) for p in names]
q75s = [ratios[p].quantile(0.75) for p in names]
ax.errorbar(x, medians, yerr=[[m - q for m, q in zip(medians, q25s)],
                               [q - m for m, q in zip(medians, q75s)]],
            fmt='o', color='black', capsize=5, markersize=7, label='Model median (IQR)')

ax.set_xticks(x)
ax.set_xticklabels([f'{n}\n({LITERATURE[n]["group"]})' for n in names], fontsize=9)
ax.set(ylabel='Pigment / TChla ratio', title='Observed Pigment Ratios vs. Literature Ranges')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

View how numerical values of pigments change over time.

In [ ]:
# --- Compute mean pigment concentration per date ---
_pig_cols = [
    c for c in pigments_df.columns if c not in ("T chla", "lon", "lat", "date")
]

_dates = pd.to_datetime(pigments_df["date"])
_grouped = pigments_df.groupby(_dates.dt.date)
_daily_mean = _grouped[_pig_cols].mean()
_daily_std = _grouped[_pig_cols].std()
_daily_count = _grouped[_pig_cols[0]].count()

_PFT_MAP = {
    "Perid":   "Dinoflagellates",
    "Fuco":    "Diatoms",
    "Zea":     "Cyanobacteria",
    "DV chla": "Cyanobacteria",
    "chl c3":  "Haptophytes",
    "HexFuco": "Haptophytes",
    "MV chlb": "Green algae",
}

# --- 3x4 grid: one time series per pigment ---
fig, axes = plt.subplots(3, 4, figsize=(18, 10), constrained_layout=True)
fig.suptitle(
    f"Eddy #{c_eddy_id} — mean pigment concentration over time",
    fontsize=16, fontweight="bold",
)

for i, pig in enumerate(_pig_cols):
    ax = axes.flat[i]
    x = _daily_mean.index
    y = _daily_mean[pig].values
    err = _daily_std[pig].values

    ax.fill_between(x, y - err, y + err, alpha=0.25, color="steelblue")
    ax.plot(x, y, "o-", color="steelblue", markersize=4, linewidth=1.5)

    # Label with PFT if diagnostic
    label = pig
    if pig in _PFT_MAP:
        label += f" \u2192 {_PFT_MAP[pig]}"
    ax.set_title(label, fontsize=9)

    ax.set_ylabel("mg/m\u00b3", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)

    # Annotate pixel count on top of each point
    for xi, yi, n in zip(x, y, _daily_count.values):
        ax.annotate(f"{n:,}", (xi, yi), textcoords="offset points",
                    xytext=(0, 8), fontsize=6, ha="center", color="gray")

plt.show()

# --- Print summary table ---
print(f"\n{'Date':<14s}  {'n_pixels':>8s}", end="")
for pig in _pig_cols:
    print(f"  {pig:>10s}", end="")
print()
print("-" * (24 + 12 * len(_pig_cols)))
for date in _daily_mean.index:
    n = _daily_count.loc[date]
    print(f"{str(date):<14s}  {n:>8,d}", end="")
    for pig in _pig_cols:
        print(f"  {_daily_mean.loc[date, pig]:>10.4f}", end="")
    print()

In [ ]:
# --- Percentile envelopes per pigment over time ---
# Shows p5, p25, median, p75, p95 as nested shaded bands ("fan chart")

_pct_levels = [5, 25, 50, 75, 95]

# Compute percentiles per date for each pigment
_daily_pcts = {}  # pigment -> {pct_level: array over dates}
for pig in _pig_cols:
    _daily_pcts[pig] = {}
    for p in _pct_levels:
        _daily_pcts[pig][p] = _grouped[pig].quantile(p / 100.0).values

_pct_dates = _daily_mean.index  # same date index as the mean cell

fig, axes = plt.subplots(3, 4, figsize=(18, 10), constrained_layout=True)
fig.suptitle(
    f"Eddy #{c_eddy_id} — pigment concentration percentiles over time",
    fontsize=16, fontweight="bold",
)

for i, pig in enumerate(_pig_cols):
    ax = axes.flat[i]

    # Outer band: p5–p95 (90% of data)
    ax.fill_between(
        _pct_dates,
        _daily_pcts[pig][5],
        _daily_pcts[pig][95],
        alpha=0.15, color="steelblue", label="p5–p95",
    )
    # Inner band: p25–p75 (IQR)
    ax.fill_between(
        _pct_dates,
        _daily_pcts[pig][25],
        _daily_pcts[pig][75],
        alpha=0.30, color="steelblue", label="p25–p75",
    )
    # Median line
    ax.plot(
        _pct_dates,
        _daily_pcts[pig][50],
        "o-", color="steelblue", markersize=3, linewidth=1.5, label="median",
    )

    # Label with PFT if diagnostic
    label = pig
    if pig in _PFT_MAP:
        label += f" \u2192 {_PFT_MAP[pig]}"
    ax.set_title(label, fontsize=9)

    ax.set_ylabel("mg/m\u00b3", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)

    # Add legend only on first subplot to avoid clutter
    if i == 0:
        ax.legend(fontsize=7, loc="upper right")

# Hide unused subplot (12 pigments = 3x4, all used, but just in case)
for j in range(len(_pig_cols), len(axes.flat)):
    axes.flat[j].set_visible(False)

plt.show()

View relative concentrations of various pigments within eddy over time.

In [ ]:
from datetime import datetime as _dt, timedelta as _td

# QC fractions from unfiltered archive (for title annotation only)
_qc_meta = pd.read_csv(
    f"archive/cyclone_track20_analysis/eddy_{c_eddy_id}_rrs.csv",
    usecols=["date", "qc_fraction"],
)
_qc_per_date = _qc_meta.groupby("date")["qc_fraction"].first()

# --- Eddy contour lookup ---
_eddy_mask = c_long.track == c_eddy_id
_eddy_clons = (c_long.contour_lon_e[_eddy_mask] + 180) % 360 - 180
_eddy_clats = c_long.contour_lat_e[_eddy_mask]
_eddy_times = c_long.time[_eddy_mask]

_PET_EPOCH = _dt(1950, 1, 1)
_date_to_idx = {
    (_PET_EPOCH + _td(days=int(t))).strftime("%Y-%m-%d"): i
    for i, t in enumerate(_eddy_times)
}

# All dates present in the (already-filtered) pigments data — no QC threshold
SNAPSHOT_DATES = sorted(pigments_df["date"].dt.strftime("%Y-%m-%d").unique())

print(
    f"eddy_id: {c_eddy_id}\n"
    f"snapshot_dates: {len(SNAPSHOT_DATES)}"
)
for d in SNAPSHOT_DATES:
    day_n = (pigments_df["date"].dt.strftime("%Y-%m-%d") == d).sum()
    qc = _qc_per_date.get(d, float("nan"))
    print(
        f"date: {d}\n"
        f"pixels: {day_n:,}\n"
        f"qc_percent: {qc:.1%}"
    )

# 12 accessory pigment columns (exclude T chla)
PIGMENT_COLS = [
    col for col in pigments_df.columns
    if col not in ("T chla", "lon", "lat", "date")
]

# Key diagnostic pigment → PFT mapping (from clustering analysis)
PFT_MAP = {
    "Perid":   "Dinoflagellates",
    "Fuco":    "Diatoms",
    "Zea":     "Cyanobacteria",
    "DV chla": "Cyanobacteria",
    "chl c3":  "Haptophytes",
    "HexFuco": "Haptophytes",
    "MV chlb": "Green algae",
}

# Percentile bounds for robust normalisation
_PCT_LO, _PCT_HI = 5, 95

# --- Generate one figure per snapshot date ---
for date_str in SNAPSHOT_DATES:
    day_pig = pigments_df[pigments_df["date"].dt.strftime("%Y-%m-%d") == date_str]
    n_pixels = len(day_pig)
    qc = _qc_per_date.get(date_str, float("nan"))

    cidx = _date_to_idx[date_str]
    clon = _eddy_clons[cidx]
    clat = _eddy_clats[cidx]
    clon_closed = np.append(clon, clon[0])
    clat_closed = np.append(clat, clat[0])

    # Reconstruct the regular grid from the sparse valid points
    unique_lon = np.unique(day_pig["lon"].values)
    unique_lat = np.unique(day_pig["lat"].values)

    # Grid spacing (L3 is uniform 4km ≈ 0.0417°)
    dlon = np.median(np.diff(unique_lon))
    dlat = np.median(np.diff(unique_lat))

    # Build a lookup from (lon, lat) → row index for fast 2D placement
    lon_to_j = {v: j for j, v in enumerate(unique_lon)}
    lat_to_i = {v: i for i, v in enumerate(unique_lat)}

    # pcolormesh cell edges: N+1 edges for N cell centers
    lon_edges = np.concatenate([unique_lon - dlon / 2, [unique_lon[-1] + dlon / 2]])
    lat_edges = np.concatenate([unique_lat - dlat / 2, [unique_lat[-1] + dlat / 2]])

    fig, axes = plt.subplots(3, 4, figsize=(18, 13), constrained_layout=True)
    fig.suptitle(
        f"Eddy #{c_eddy_id} pigment concentrations — {date_str}   "
        f"({n_pixels:,} pixels, QC={qc:.1%})",
        fontsize=16,
        fontweight="bold",
    )

    axes_flat = axes.flatten()

    for i, pigment in enumerate(PIGMENT_COLS):
        ax = axes_flat[i]
        vals = day_pig[pigment].values

        # Percentile-based normalisation: clip then scale to [0, 1]
        p5 = np.percentile(vals, _PCT_LO)
        p95 = np.percentile(vals, _PCT_HI)
        if p95 - p5 > 0:
            clipped = np.clip(vals, p5, p95)
            norm_vals = (clipped - p5) / (p95 - p5)
        else:
            norm_vals = np.full_like(vals, 0.5)

        # Place normalised values into a 2D grid (NaN where no valid pixel)
        grid = np.full((len(unique_lat), len(unique_lon)), np.nan)
        for nv, lon_val, lat_val in zip(
            norm_vals, day_pig["lon"].values, day_pig["lat"].values
        ):
            grid[lat_to_i[lat_val], lon_to_j[lon_val]] = nv

        pc = ax.pcolormesh(
            lon_edges, lat_edges, grid,
            cmap="RdYlBu_r",
            vmin=0,
            vmax=1,
            rasterized=True,
        )

        ax.plot(clon_closed, clat_closed, color="black", linewidth=1.5)

        label = pigment
        if pigment in PFT_MAP:
            label += f" → {PFT_MAP[pigment]}"
        ax.set_title(f"{label}\n[p5={p5:.4f}  p95={p95:.4f} mg/m³]", fontsize=9)
        ax.set_aspect("equal")
        ax.tick_params(labelsize=7)

        if i % 4 == 0:
            ax.set_ylabel("Latitude", fontsize=8)
        if i >= 8:
            ax.set_xlabel("Longitude", fontsize=8)

    cbar = fig.colorbar(pc, ax=axes, location="right", shrink=0.8, pad=0.02)
    cbar.set_label("Normalised concentration (0 = p5, 1 = p95)", fontsize=10)

    plt.show()

### Translate phytoplankton pigments to phytoplankton groups

Two primary approaches:
- CHEMTAX
    - Assumes ratio of a specific pigment to total chlorophyll a for a given phytoplankton group is fixed/known.
    - Assumes linear independence between pigments, but multicolinearity could mess up the matrix inversion process.
- Hierarchical clustering + empirical orthogonal functions
    - https://pmc.ncbi.nlm.nih.gov/articles/PMC7043335/
    

For any 2 branches connected by a horizontal link, the height of the horizontal segment on the y-axis is the distance. The lower two are, the more similar.

In [ ]:
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform

keep_cols = [c for c in pigments_df.columns if c not in ['T chla', 'lon', 'lat', 'date']]

ratios = pigments_df[keep_cols].div(pigments_df['T chla'], axis=0) # 12 cols

corr = ratios.corr() # 12x12 symmetric matrix
dist = 1 - corr
condensed = squareform(dist, checks=False)
linkage = sch.linkage(condensed, method='ward')

clusters = sch.fcluster(linkage, t=0.65, criterion='distance')

fig, ax = plt.subplots(figsize=(10, 6))
sch.dendrogram(linkage, labels=keep_cols, ax=ax, color_threshold=0.65)
plt.show()



- **Diatoms**: Fuco + chl c1+c2
- **Haptophytes**: HexFuco + ButFuco + chl c3
- **Cyanobacteria**: Zea + DV chla
- **Dinoflagellates**: Perid
- **Green algae**: MV chlb + Viola + Neo
- **Cryptophytes**: Allo

Kramer 2019 also found that allo (cryptophytes) clusters with green algae (right side).
Not sure about HexFuco and ButFuco being with Perid.

**Obtaining a numerical value for PFT abundance.**

We start with a bunch of pigment concentrations, and we want to know what PFT combination is responsible for this. Each PFT produces a relative ratio of pigment to tchla, and the F-matrix encodes this information where the cols are pigments and rows are PFTs, with values at each cell representing pigment:tchla ratio for that PFT.

For each observation:

$$p = xF$$
- p is the known pigments concentrations (d, )
- x is what we solve for: vector of tchla contributed by each class (c, )
- F is matrix of pigment:tchla ratio per class (c, d)

The issue is that $x$ is unknown and $F$ is only partially known (only have prior belief initially). 

By phytoclass (https://aslopubs.onlinelibrary.wiley.com/doi/full/10.1002/lom3.10541), this F-matrix starts out as binary encoding (1 if this pigment is known to be assiciated with a PFT class otherwise 0).

The inner loop runs simulated annealing, which is a probabilistic method for finding an optimal solution to a function. This is done by picking some random F, then running non-negative least-squares to measure RMSE. Better solutions are always accepted, and worse solutions are accepted with a probability that decreases with temperature drops. (Temperature goes from high -> low over iterations).

After obtaining the F matrix, we then run non-negative last-squares again to obtain x (overdetermined).

In [ ]:
# (n_obs, n_pft)
pfts_df = pd.read_csv('/Users/jerry/school/research/eddy-tracking/archive/cyclone_track20_analysis/eddy_20_pfts.csv')
pfts_df.describe()

In [ ]:
# Marks contribution percentage to total chlorophyll

row_sums = pfts_df.sum(axis=1)
tchla_ratios = pfts_df.div(row_sums, axis=0).fillna(0)
mean_ratios = tchla_ratios.mean(axis=0)
print(mean_ratios * 100)

print("\n\nTotal:")
print(mean_ratios.sum())

Groups that should be abundant near the gulf stream:
- Haptophytes
- Cyanobacteria
- Diatoms, but only when there is an abundance of nutrients
- Cryptophytes should be least abundant

**Next steps**

Quality Verification:
- Some days have partial QC flags; while the sensors may not have flagged other nearby pixels as having issues, this model seems fairly sensitive to Rrs changes.
- Look for days where TChla predictions are exactly 0, and where pixels have negative Rrs values
- Look for the peaks and lows in the mean/percentile plots
    - For example, mid-March for Perid, 4/4, 4/16

Moving from pigment clusters / PFT groups to a quantifiable metric. Like a percentage composition, or some raw number indicating abundance given pigment concentrations.
- What existing methods are there?
- What factors influence this translation? Season? Lighting conditions? Geographic region?

Also curious about the methods involved from the Rrs -> pigment step and pigment -> phytoplankton step:
- Using tchla as a predictor, since pigment concentrations clearly vary depending on environment (oligotrophic vs. nutrient-rich coastal waters). Open ocean is fine, but in coastal waters where results are much more impactful research is doing worse relatively.

In [ ]:
# here